In [40]:
# Import required libraries
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

print("Libraries imported successfully!")


Libraries imported successfully!


In [41]:
# Database connection configuration to SAKIP
DB_HOST = os.getenv('DB_HOST_SAKIP', 'localhost')
DB_PORT = os.getenv('DB_PORT_SAKIP', '5432')  # PostgreSQL default port
DB_NAME = os.getenv('DB_DATABASE_SAKIP', 'your_database_name')
DB_USER = os.getenv('DB_USERNAME_SAKIP', 'your_username')
DB_PASSWORD = os.getenv('DB_PASSWORD_SAKIP', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"Connecting to database: {DB_NAME} on {DB_HOST}:{DB_PORT}")
print(f"User: {DB_USER}")

# Test connection
try:
    engine_sakip = create_engine(connection_string, echo=False)
    with engine_sakip.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: e-sakip on 10.110.34.49:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [42]:
def load_data_from_sql(query, engine):
    """
    Load data from PostgreSQL database into a pandas DataFrame.
    
    Parameters:
    query (str): SQL query to execute
    engine: SQLAlchemy engine object
    
    Returns:
    pandas.DataFrame: Data from the query
    """
    connection = None
    try:
        # Create a new connection and rollback any pending transaction
        connection = engine.connect()
        
        # Rollback any pending transaction to ensure clean state
        try:
            connection.rollback()
        except:
            pass  # If no transaction to rollback, ignore
        
        # Execute the query
        df = pd.read_sql(query, connection)
        print(f"✅ Data loaded successfully! Shape: {df.shape}")
        
        connection.close()
        return df
    except Exception as e:
        # Ensure connection is closed on error
        if connection:
            try:
                connection.rollback()
            except:
                pass
            try:
                connection.close()
            except:
                pass
        print(f"❌ Error loading data: {e}")
        return None

def get_table_info(table_name, engine):
    """
    Get basic information about a table.
    
    Parameters:
    table_name (str): Name of the table
    engine: SQLAlchemy engine object
    """
    try:
        # Get table structure using PostgreSQL information_schema
        structure_query = f"""
            SELECT 
                column_name,
                data_type,
                character_maximum_length,
                is_nullable,
                column_default
            FROM information_schema.columns
            WHERE table_name = '{table_name}'
            ORDER BY ordinal_position
        """
        structure = pd.read_sql(structure_query, engine)
        
        # Get row count
        count_query = f"SELECT COUNT(*) as row_count FROM {table_name}"
        count_result = pd.read_sql(count_query, engine)
        
        print(f"📊 Table: {table_name}")
        print(f"Rows: {count_result['row_count'].iloc[0]}")
        print(f"Columns: {len(structure)}")
        print("\nColumn Information:")
        print(structure)
        
        return structure
    except SQLAlchemyError as e:
        print(f"❌ Error getting table info: {e}")
        return None

def list_tables(engine):
    """
    List all tables in the database.
    
    Parameters:
    engine: SQLAlchemy engine object
    """
    try:
        # Use PostgreSQL information_schema to list tables
        query = """
            SELECT table_name 
            FROM information_schema.tables 
            WHERE table_schema = 'public'
            ORDER BY table_name
        """
        tables = pd.read_sql(query, engine)
        print("📋 Available tables:")
        for table in tables['table_name']:
            print(f"  - {table}")
        return tables
    except SQLAlchemyError as e:
        print(f"❌ Error listing tables: {e}")
        return None

print("Data loading functions defined successfully!")



Data loading functions defined successfully!


In [43]:
query = "SELECT * FROM skp \
WHERE tahun_kinerja = '2026' \
AND is_skp;"

df_skp = pd.DataFrame()

df_skp = load_data_from_sql(query, engine_sakip)

✅ Data loaded successfully! Shape: (7228, 17)


In [44]:
df_skp.head()

,id,tahun_mulai,tahun_kinerja,satuan_kerja_id,pengampu,v_struktur_organisasi_id,tim_kerja_id,model_class,model_id,sasaran,indikator,target,satuan,created_at,updated_at,parent_id,is_skp
0,41652,2025,2026,1003,tim-kerja,None,18047.0,App\Models\KinerjaSubKegiatan,52104,Teridentifikasinya data calon peserta murid baru,Jumlah data calon peserta didik baru,1,dokumen,2026-01-31 20:36:53,2026-01-31 20:36:53,36067.0,True
1,36525,2025,2026,1002,tim-kerja,None,11584.0,App\Models\KinerjaSubKegiatan,45073,Tersusunnya bahan rapat DPRD,jumlah bahan rapat DPRD,2,Dokumen,2026-01-31 20:35:47,2026-01-31 20:35:47,36110.0,True
2,36526,2025,2026,1002,tim-kerja,None,11585.0,App\Models\KinerjaSubKegiatan,45074,Tersusunnya berita acara rapat,Jumlah berita acara rapat,2,Dokumen,2026-01-31 20:35:47,2026-01-31 20:35:47,36110.0,True
3,40810,2025,2026,1020,tim-kerja,None,17886.0,App\Models\KinerjaSubKegiatan,50945,Tersalurkannya Sumur Dalam ditingkat usaha tan...,Jumlah Poktan Tembakau yang menerima bantuan S...,1,Poktan,2026-01-31 20:36:42,2026-01-31 20:36:42,34799.0,True
4,40811,2025,2026,1020,tim-kerja,None,17217.0,App\Models\KinerjaSubKegiatan,50948,Terbinanya penyuluh pertanian tentang perkebunan,Jumlah penyuluh pertanian yang terbina tentang...,20,orang,2026-01-31 20:36:42,2026-01-31 20:36:42,34858.0,True


In [45]:
# Collect tim_kerja_id 

tim_kerja_ids = [str(int(x)) for x in df_skp['tim_kerja_id'].unique() if pd.notnull(x)]

len(tim_kerja_ids)

3423

In [46]:
tim_kerja_ids 

['18047',
 '11584',
 '11585',
 '17886',
 '17217',
 '17413',
 '17052',
 '18484',
 '17065',
 '18005',
 '18012',
 '18116',
 '17105',
 '56',
 '17107',
 '18704',
 '17889',
 '17299',
 '17111',
 '16965',
 '17478',
 '17580',
 '17566',
 '18951',
 '20045',
 '20046',
 '18957',
 '17121',
 '17123',
 '12025',
 '19228',
 '18016',
 '17177',
 '19790',
 '18143',
 '18765',
 '19046',
 '18792',
 '17292',
 '17866',
 '17281',
 '17255',
 '17257',
 '17874',
 '19796',
 '17575',
 '19089',
 '15715',
 '251',
 '18534',
 '15738',
 '16933',
 '16932',
 '10866',
 '19634',
 '18789',
 '19631',
 '19791',
 '16957',
 '18634',
 '16587',
 '19723',
 '19254',
 '18368',
 '16757',
 '19246',
 '12639',
 '17523',
 '19242',
 '19244',
 '19331',
 '15751',
 '19279',
 '18485',
 '17481',
 '15473',
 '19230',
 '19236',
 '17482',
 '18980',
 '19660',
 '17137',
 '18483',
 '19281',
 '12610',
 '17153',
 '17157',
 '16580',
 '17160',
 '19764',
 '17168',
 '17702',
 '17176',
 '17483',
 '18616',
 '17582',
 '14960',
 '18788',
 '17170',
 '17171',
 '185

In [47]:
# Database connection configuration to TRK 2026
DB_HOST_2026 = os.getenv('DB_HOST_2026', 'localhost')
DB_PORT_2026 = os.getenv('DB_PORT_2026', '5432')  # PostgreSQL default port
DB_NAME_2026 = os.getenv('DB_DATABASE_2026', 'your_database_name')
DB_USER_2026 = os.getenv('DB_USERNAME_2026', 'your_username')
DB_PASSWORD_2026 = os.getenv('DB_PASSWORD_2026', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER_2026}:{DB_PASSWORD_2026}@{DB_HOST_2026}:{DB_PORT_2026}/{DB_NAME_2026}"

print(f"Connecting to database: {DB_NAME_2026} on {DB_HOST_2026}:{DB_PORT_2026}")
print(f"User: {DB_USER_2026}")

# Test connection
try:
    engine_2026 = create_engine(connection_string, echo=False)
    with engine_2026.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: erk_ekinerja_2026 on 10.110.32.114:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [48]:
# Collect data from tim_kerja
query = "SELECT * FROM tim_kerja \
WHERE id IN ('" + "', '".join(tim_kerja_ids) + "')"

df_tim_kerja = pd.DataFrame()

df_tim_kerja = load_data_from_sql(query, engine_2026)


✅ Data loaded successfully! Shape: (3423, 8)


In [49]:
df_tim_kerja.head()

,id,nama,satuan_kerja_id,v_struktur_organisasi_id,nip_ketua,created_at,updated_at,deleted_at
0,54,Pencegahan Dampak Lingkungan,1034,103407030000,197306121998032013,2022-01-31 13:17:50,2022-01-31 13:17:50,None
1,56,"Pengendalian Sampah, B3 dan Limbah B3",1034,103407030000,197807302006042005,2022-01-31 13:36:30,2022-01-31 13:36:30,None
2,58,Pengendalian Pencemaran Air dan Udara,1034,103407030000,197701142009011001,2022-01-31 13:46:00,2022-01-31 13:46:00,None
3,59,Konservasi Lingkungan dan Keanekaragaman Hayati,1034,103407030000,198007152011012001,2022-01-31 13:55:02,2022-01-31 13:55:02,None
4,60,Pengembangan Kapasitas dan Kemitraan,1034,103407030000,197909122011012001,2022-01-31 13:58:22,2022-01-31 13:58:22,None


In [63]:
# Collect nip_ketua
nip_ketuas = df_tim_kerja['nip_ketua'].unique()

nip_ketuas.tolist()

['197306121998032013',
 '197807302006042005',
 '197701142009011001',
 '198007152011012001',
 '197909122011012001',
 '197908122006042006',
 '197309121998031004',
 '198008112010011002',
 '198011142011012003',
 '197306282008011003',
 '196903172002121001',
 '196806141997031004',
 '198201122001121004',
 '198001262011011002',
 '199208142014061001',
 '197902212008012003',
 '197603112014122001',
 '198010202010011003',
 '197103022005011010',
 '198211132008082001',
 '197606162008011004',
 '197011251998031005',
 '198609202010012006',
 '197504281999022001',
 '197902162011011001',
 '197410032008012003',
 '198007042011011001',
 '197411241999031005',
 '196902261989031004',
 '197402152006041009',
 '197409032009011001',
 '197108101997022005',
 '197005231998031007',
 '197608162005011011',
 '197904102011011002',
 '196705171990111001',
 '197410262008012003',
 '197601242006042004',
 '198108232005011008',
 '197709022009012001',
 '199709232020121009',
 '197606162009011004',
 '197707282005011006',
 '198209042

In [62]:
nip_jpts_str = '197011111991021001,196904201988032004,196906121988031002,196812161998031003,197010151991031004,197606121996031005,197306171993031004,197912182005011008,196704211992031013,198311172006041009,196704061994022002,197603021997011001,198109252010011010,198202142006041004,197306241998031008,197804191996121001,197307071999022001,198409112002121001,197309182005011004,197302222001121003,197108252005012009,197808112003121005,196607101993031005,197404051998031012,197812222006042009,196610201998031003,197201162002122002,196706301998032005,198008292000031002,197409261998032005,197503171999012001,197204191998031007,196610271987021002,197709172002122004,197203181998031007,196710111993031009,197403051997031003,197808062008011003,197211061999012001,197805052005011016,196703191994031001,197212181998031008,196609111994022001,197108131997031007,196707291993031004,196804041988032014,197909171998101001,197312121993031002,197009221998031004,197108121998032006,196812221996011001,196906232006041004,197509132006041008,197906281998101001'

nip_jpts_list = nip_jpts_str.split(',')

nip_jpts_list

['197011111991021001',
 '196904201988032004',
 '196906121988031002',
 '196812161998031003',
 '197010151991031004',
 '197606121996031005',
 '197306171993031004',
 '197912182005011008',
 '196704211992031013',
 '198311172006041009',
 '196704061994022002',
 '197603021997011001',
 '198109252010011010',
 '198202142006041004',
 '197306241998031008',
 '197804191996121001',
 '197307071999022001',
 '198409112002121001',
 '197309182005011004',
 '197302222001121003',
 '197108252005012009',
 '197808112003121005',
 '196607101993031005',
 '197404051998031012',
 '197812222006042009',
 '196610201998031003',
 '197201162002122002',
 '196706301998032005',
 '198008292000031002',
 '197409261998032005',
 '197503171999012001',
 '197204191998031007',
 '196610271987021002',
 '197709172002122004',
 '197203181998031007',
 '196710111993031009',
 '197403051997031003',
 '197808062008011003',
 '197211061999012001',
 '197805052005011016',
 '196703191994031001',
 '197212181998031008',
 '196609111994022001',
 '197108131

In [53]:
# Collect non null v_struktur_organisasi_id
v_struktur_organisasi_ids = df_tim_kerja['v_struktur_organisasi_id'].dropna().unique()

len(v_struktur_organisasi_ids)

399

In [54]:
# Database connection configuration to SIMPEG
DB_HOST_SIMPEG = os.getenv('DB_HOST_SIMPEG', 'localhost')
DB_PORT_SIMPEG = os.getenv('DB_PORT_SIMPEG', '5432')  # PostgreSQL default port
DB_NAME_SIMPEG = os.getenv('DB_DATABASE_SIMPEG', 'your_database_name')
DB_USER_SIMPEG = os.getenv('DB_USERNAME_SIMPEG', 'your_username')
DB_PASSWORD_SIMPEG = os.getenv('DB_PASSWORD_SIMPEG', 'your_password')

# Create connection string for PostgreSQL
connection_string = f"postgresql+psycopg2://{DB_USER_SIMPEG}:{DB_PASSWORD_SIMPEG}@{DB_HOST_SIMPEG}:{DB_PORT_SIMPEG}/{DB_NAME_SIMPEG}"

print(f"Connecting to database: {DB_NAME_SIMPEG} on {DB_HOST_SIMPEG}:{DB_PORT_SIMPEG}")
print(f"User: {DB_USER_SIMPEG}")

# Test connection
try:
    engine_simpeg = create_engine(connection_string, echo=False)
    with engine_simpeg.connect() as connection:
        result = connection.execute(text("SELECT 1 as test"))
        print("✅ Database connection successful!")
        print(f"Connection test result: {result.fetchone()[0]}")
except SQLAlchemyError as e:
    print(f"❌ Database connection failed: {e}")
    print("Please check your database credentials in the .env file")


Connecting to database: simpeg_jabar on 10.110.32.121:5432
User: postgres
✅ Database connection successful!
Connection test result: 1


In [56]:

query = (
    "SELECT * FROM m_spg_jabatan "
    "WHERE unit_kerja_id IN ('" + "', '".join(v_struktur_organisasi_ids) + "') "
    "AND jfu_id is null AND jf_id is null"
)

df_m_spg_jabatan = pd.DataFrame()

df_m_spg_jabatan = load_data_from_sql(query, engine_simpeg)

df_m_spg_jabatan.head()



✅ Data loaded successfully! Shape: (381, 20)


,jabatan_id,jfu_id,satuan_kerja_id,unit_kerja_id,jabkat_id,gol_id_awal,gol_id_akhir,jf_id,eselon_id,jabatan_nama,jabatan_jenis,jabatan_type,kode_jabatan,jabatan_kelas,jabatan_bup,created_at,updated_at,create_username,update_username,deleted_at
0,10003,None,1001,100101010000,None,None,None,None,22,KEPALA BIRO PEMERINTAHAN DAN OTONOMI DAERAH,2,None,3.07.01.01.01,14,60,NaT,2019-05-27 03:02:28.517172,None,None,NaT
1,10004,None,1001,100101010100,None,None,None,None,31,KEPALA BAGIAN PEMERINTAHAN,2,None,3.07.01.01.01.01,None,58,NaT,2019-05-27 03:02:28.517172,None,exclude_distribusi_skp,2023-12-18
2,10008,None,1001,100101010200,None,None,None,None,31,KEPALA BAGIAN OTONOMI DAERAH,2,None,3.07.01.01.01.02,None,58,NaT,2019-05-27 03:02:28.517172,None,exclude_distribusi_skp,2023-12-18
3,100041205,None,1001,100101010400,None,None,None,None,31,KEPALA BAGIAN TATA USAHA,2,None,3.07.01.01.01.04,12,58,NaT,2022-08-19 04:32:39.650408,None,None,NaT
4,10016,None,1001,100101020000,None,None,None,None,22,KEPALA BIRO HUKUM DAN HAK ASASI MANUSIA,2,None,3.07.01.01.02,14,60,NaT,2019-05-27 03:02:28.517172,None,None,NaT


In [ ]:
# Collect jabatan_id
jabatan_ids = df_m_spg_jabatan['jabatan_id'].unique()

len(jabatan_ids)

381

In [59]:
# Collect pegawais with jabatan_id
query = "SELECT * FROM spg_pegawai \
WHERE jabatan_id IN ('" + "', '".join(str(x) for x in jabatan_ids) + "')"

df_spg_pegawai = pd.DataFrame()

df_spg_pegawai = load_data_from_sql(query, engine_simpeg)

df_spg_pegawai.head()


✅ Data loaded successfully! Shape: (581, 118)


,peg_id,id_goldar,gol_id_awal,id_pend_awal,unit_kerja_id,kecamatan_id,gol_id_akhir,jabatan_id,id_pend_akhir,id_agama,...,peg_pppk_tmt,sk_cpns_tanggal,sk_pns_tanggal,peg_kk,no_sk_udin,no_sk_pi,sk_udin_tanggal,sk_pi_tanggal,peg_no_bpjs_ketenagakerjaan,formasi
0,198409112002121001,4.0,131.0,136.0,100101010000,None,142,10003,12,1.0,...,None,2003-01-02,2004-01-02,3273070202150014,None,None,None,None,None,None
1,197306241998031008,4.0,131.0,293.0,100101020000,None,143,10016,12,1.0,...,None,1998-04-20,1999-04-12,3217020103070001,None,None,None,None,None,None
2,197804191996121001,1.0,131.0,293.0,100101030000,None,143,10029,12,1.0,...,None,None,None,None,None,None,None,None,None,None
3,197302222001121003,2.0,131.0,293.0,100102010000,None,142,10043,12,1.0,...,None,None,None,None,None,None,None,None,None,None
4,198202142006041004,1.0,131.0,293.0,100102020000,None,141,10056,12,1.0,...,None,2006-07-20,2007-09-27,3273161309103677,None,None,None,None,None,None


In [64]:
# Collect nips
nips = df_spg_pegawai['peg_nip'].unique()

nips.tolist()

['198409112002121001',
 '197306241998031008',
 '197804191996121001',
 '197302222001121003',
 '198202142006041004',
 '196801101990031001',
 '197309182005011004',
 '197307071999022001',
 '195901031985031009',
 '198109252010011010',
 '198608112006021001',
 '196803301994031003',
 '198403192009022002',
 '198801062007012001',
 '196711161994031004',
 '196512231990021001',
 '197002281998021002',
 '196501031989031007',
 '197710022009021003',
 '196904091994021001',
 '196312081985031011',
 '197402041993111002',
 '198212282002121002',
 '196407101993031009',
 '197103011998021002',
 '196611151991011002',
 '196503071985122001',
 '196606301988111002',
 '196804071998021002',
 '198408212009011003',
 '196101221981092001',
 '198409072002121001',
 '197001221993032003',
 '198302052002121001',
 '196103281982021002',
 '196506181996031001',
 '196609281993032004',
 '196909111992012001',
 '196806131994032004',
 '196604281987121002',
 '197612022003121004',
 '196709141994031004',
 '197308172000031010',
 '196108151

In [ ]:
# Combine three lists into one
all_nips = pd.unique(list(nip_ketuas) + list(nip_jpts_list) + list(nips)).tolist()

all_nips


['197306121998032013',
 '197807302006042005',
 '197701142009011001',
 '198007152011012001',
 '197909122011012001',
 '197908122006042006',
 '197309121998031004',
 '198008112010011002',
 '198011142011012003',
 '197306282008011003',
 '196903172002121001',
 '196806141997031004',
 '198201122001121004',
 '198001262011011002',
 '199208142014061001',
 '197902212008012003',
 '197603112014122001',
 '198010202010011003',
 '197103022005011010',
 '198211132008082001',
 '197606162008011004',
 '197011251998031005',
 '198609202010012006',
 '197504281999022001',
 '197902162011011001',
 '197410032008012003',
 '198007042011011001',
 '197411241999031005',
 '196902261989031004',
 '197402152006041009',
 '197409032009011001',
 '197108101997022005',
 '197005231998031007',
 '197608162005011011',
 '197904102011011002',
 '196705171990111001',
 '197410262008012003',
 '197601242006042004',
 '198108232005011008',
 '197709022009012001',
 '199709232020121009',
 '197606162009011004',
 '197707282005011006',
 '198209042

In [69]:
# Make single string separated by comma
nip_all_str = ','.join(all_nips)

nip_all_str

'197306121998032013,197807302006042005,197701142009011001,198007152011012001,197909122011012001,197908122006042006,197309121998031004,198008112010011002,198011142011012003,197306282008011003,196903172002121001,196806141997031004,198201122001121004,198001262011011002,199208142014061001,197902212008012003,197603112014122001,198010202010011003,197103022005011010,198211132008082001,197606162008011004,197011251998031005,198609202010012006,197504281999022001,197902162011011001,197410032008012003,198007042011011001,197411241999031005,196902261989031004,197402152006041009,197409032009011001,197108101997022005,197005231998031007,197608162005011011,197904102011011002,196705171990111001,197410262008012003,197601242006042004,198108232005011008,197709022009012001,199709232020121009,197606162009011004,197707282005011006,198209042009011001,198611062004122002,198702232011012002,197810212011012002,197109281999032008,196903102007011010,196906291991021001,199705302020121011,197311052003122004,19800804201

In [70]:
# Save in text file
with open('all_nips.txt', 'w') as file:
    file.write(nip_all_str)
